# Huấn luyện mô hình YOLOv8 trên Kaggle

Notebook này được tạo ra để huấn luyện mô hình YOLOv8 nhận diện tiền Việt Nam với bộ dataset `Nhan-Dien-Tien-VND1`.

## Hướng dẫn chuẩn bị trên Kaggle:
1. **Bật GPU:** Góc phải trên cùng của Kaggle, nhấp vào dấu ba chấm hoặc Session options -> **Accelerator** -> Chọn **GPU T4 x2** hoặc **GPU P100**.
2. **Thêm Dataset:**
   - Nếu bạn đã tải bộ dataset về máy, hãy nén thư mục `Nhan-Dien-Tien-VND1` thành file `.zip`.
   - Trên Kaggle, nhấn **Add Input** (Góc phải) -> **Upload a Dataset** -> Tải file zip lên và đặt tên (ví dụ: `nhandientienvnd1`).
   - Kaggle sẽ tự giải nén thư mục của bạn vào đường dẫn `/kaggle/input/...`
3. Nếu bạn muốn tải trực tiếp dataset bằng code của Roboflow thì có thể sử dụng snippet code cung cấp trên Roboflow thay vì upload (xem thêm chi tiết trên trang web roboflow).

In [1]:
# Cài đặt thư viện Ultralytics (chứa YOLOv8)
!pip install ultralytics -q
!pip install pyyaml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
import yaml
import shutil
from ultralytics import YOLO

# Hiển thị thông tin kiểm tra GPU (Đảm bảo đã được gán GPU)
!nvidia-smi

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Fri May 15 18:14:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla 

In [3]:
# Cấu hình đường dẫn tới Dataset
# TODO: Bạn có thể duyệt tìm đường dẫn chính xác ở thanh "Data" (Input) bên phải màn hình Kaggle.
DATASET_ROOT = "/kaggle/input/datasets/dungnguyen2301/dataset-vnmoney" 

# Hàm tìm tự động file data.yaml trong các thư mục input của Kaggle
def find_data_yaml(start_dir='/kaggle/input'):
    for root, dirs, files in os.walk(start_dir):
        if 'data.yaml' in files:
            return os.path.join(root, 'data.yaml')
    return None

yaml_path = find_data_yaml()
if yaml_path:
    print(f"Đã tự động tìm thấy file cấu hình tại: {yaml_path}")
    DATASET_ROOT = os.path.dirname(yaml_path)
else:
    print("CẢNH BÁO: Không tìm thấy file data.yaml! Vui lòng kiểm tra lại phần Add Input dataset của bạn.")

Đã tự động tìm thấy file cấu hình tại: /kaggle/input/datasets/dungnguyen2301/dataset-vnmoney/data.yaml


In [4]:
# Đọc và cấu hình lại đường dẫn trong file data.yaml cho phù hợp với môi trường Kaggle
# Kaggle quy định thư mục /kaggle/input/ là Read-only, vì thế ta phải đưa cấu hình về /kaggle/working/
working_yaml_path = '/kaggle/working/data.yaml'

if yaml_path:
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data_config = yaml.safe_load(f)
    
    print("Cấu hình data.yaml gốc:", data_config)
    
    # Cập nhật đường dẫn gốc (path) trỏ tới thư mục chứa dữ liệu dataset
    data_config['path'] = DATASET_ROOT
    
    # Lưu file yaml mới vào thư mục working để YOLOv8 có quyền đọc/ghi
    with open(working_yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_config, f, sort_keys=False)
        
    print(f"\nĐã tạo thành công cấu hình data mới tại: {working_yaml_path}")

Cấu hình data.yaml gốc: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['100k', '10k', '200k', '20k', '500k', '50k'], 'roboflow': {'workspace': 'detect-money', 'project': 'nhan-dien-tien-vnd1', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/detect-money/nhan-dien-tien-vnd1/dataset/1'}}

Đã tạo thành công cấu hình data mới tại: /kaggle/working/data.yaml


In [5]:
# Khởi tạo mô hình YOLOv8
# YOLOv8 có nhiều phiên bản: 'yolov8n.pt' (nano, nhanh nhất), 'yolov8s.pt' (small), 'yolov8m.pt' (medium), ...
# Dùng bản Nano (n) thường đủ tốt cho ứng dụng chạy thực tế với độ trễ thấp.
model = YOLO('yolov8n.pt') 

In [6]:
# Bắt đầu quá trình huấn luyện (Training)
results = model.train(
    data=working_yaml_path,  # File cấu hình data ta vừa chỉnh sửa
    epochs=50,               # Số vòng huấn luyện (Bạn có thể tăng lên 100 nếu mô hình chưa hội tụ)
    imgsz=640,               # Kích thước ảnh đầu vào (Thường mặc định tải từ Roboflow là 640x640)
    batch=16,                # Batch size (Kích thước mỗi lần đưa ảnh vào huấn luyện. Nếu GPU Kaggle báo lỗi OOM, hãy giảm xuống 8)
    project='/kaggle/working/runs',
    name='train_vnd_model',
    device=0,                # Sử dụng card đồ họa GPU
    patience=15              # Early stopping: Dừng sớm nếu mô hình không cải thiện thêm sau 15 epoch để tránh overfitting
)

Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_vnd_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

In [7]:
# Đánh giá mô hình trên tập validation sau khi train xong
metrics = model.val(split='test')
print(f"Độ chính xác trung bình (mAP50): {metrics.box.map50}")
print(f"Độ chính xác trung bình (mAP50-95): {metrics.box.map}")

Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5.1±2.5 MB/s, size: 19.8 KB)
val: Scanning /kaggle/input/datasets/dungnguyen2301/dataset-vnmoney/test/labels... 977 images, 171 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 977/977 295.0it/s 3.3s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/dungnguyen2301/dataset-vnmoney/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 8.7it/s 7.2s<0.2s
                   all        977        807      0.828      0.768      0.806      0.401
                  100k        130        130      0.845      0.711      0.714      0.295
                   10k        136        137       0.84      0.766      0.808      0.311
                  200k        136        136      0.744      0.78

#### Xuất mô hình tốt nhất (best.pt) ra thư mục /kaggle/working để dễ dàng download về máy cục bộ
best_model_path = '/kaggle/working/runs/train_vnd_model/weights/best.pt'
export_path = '/kaggle/working/best_vnd_model.pt'

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, export_path)
    print(f"\nĐÃ HOÀN TẤT! Bạn có thể download mô hình tốt nhất tại file: {export_path}")
    print("Vui lòng kiểm tra ở mục 'Output' bên thanh công cụ tay phải của Kaggle để tải xuống file best_vnd_model.pt")
else:
    print("Không tìm thấy file mô hình best.pt.")